# 异步编程

学习目标：用协程组织可等待的工作，在任务组合、超时和取消时保留正确的结果传播与资源清理，并控制并发数量和任务上下文。

前置知识：函数、生成器、迭代协议、with、异常组与 except*、线程与 Future、对象引用。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

IPykernel 已运行事件循环，本章直接使用顶层 await、async for 和 async with。独立 Python 脚本通常由 asyncio.run 调用入口协程；不要在这个内核的同一线程中再次调用它。

## 1 协程与 await

### 1.1 调用 async def 先得到协程对象

async def 定义协程函数（coroutine function），调用后得到协程对象（coroutine object），函数体不会仅因调用而运行。await 执行或等待一个可等待对象（awaitable），并取得结果；创建后既不等待也不调度的协程可能产生未等待警告。

| 名称 | 中文名称／含义 |
| --- | --- |
| coroutine | 协程对象；由协程函数调用产生 |
| asyncio.Task | 任务；负责调度一个协程并保存最终状态 |
| asyncio.Future | 可等待的最终结果，通常由底层库提供 |
| event loop | 事件循环；调度任务、回调和 I/O 就绪事件 |

这里的 asyncio.Task 可以被 await；线程池的 concurrent.futures.Future 使用另一套接口，不能把两者当成同一对象。

In [1]:
import asyncio


async def read_minutes(trace: list[str]) -> int:
    """记录函数体实际开始的时机，并返回固定的学习时长。"""
    trace.append("开始读取")
    return 25


coroutine_trace = []
minutes_coroutine = read_minutes(coroutine_trace)
print(coroutine_trace)  # []：只有协程对象，函数体还没运行。
print(await minutes_coroutine)  # 25：等待时才执行函数体。
print(coroutine_trace)  # ['开始读取']：没有遗留未等待的协程。

[]
25
['开始读取']


### 1.2 连续 await 默认保持顺序

在当前协程里先 await 第一个调用，再 await 第二个调用，第二个要等第一个返回才开始。写了 async def 不等于每次调用都会自动并发。

asyncio.sleep(0) 会挂起当前任务，让事件循环有机会调度其他工作。本章仅把它用作明确的让出点，不用休眠时长猜测两个任务的先后。

In [2]:
async def prepare_label(label: str, trace: list[str]) -> str:
    """让出一次执行机会，记录标签处理的开始与结束。"""
    trace.append(f"{label}开始")
    await asyncio.sleep(0)
    trace.append(f"{label}结束")
    return label.upper()


sequential_trace = []
first_label = await prepare_label("a", sequential_trace)
second_label = await prepare_label("b", sequential_trace)
print(first_label, second_label)  # A B
print(sequential_trace)  # ['a开始', 'a结束', 'b开始', 'b结束']

A B
['a开始', 'a结束', 'b开始', 'b结束']


## 2 事件循环与阻塞边界

### 2.1 等待事件时让其他任务继续

同一个事件循环线程在某一时刻运行一个任务；任务等待尚未完成的操作时，循环才有机会执行其他工作。这种配合调度称为协作式调度（cooperative scheduling）。

asyncio.Event 用于任务间通知，wait 是需要 await 的操作。下面用 started 确认工作任务进入等待，再由 gate 放行；asyncio 的同步原语不是线程安全工具。

In [3]:
async def wait_for_gate(
    started: asyncio.Event,
    gate: asyncio.Event,
    cleanup: list[str],
) -> str:
    """等待任务间通知，并在正常或取消退出时记录清理。"""
    started.set()
    try:
        await gate.wait()
        return "已放行"
    finally:
        cleanup.append("已清理")


started = asyncio.Event()
gate = asyncio.Event()
cleanup = []
# 先创建任务，再等待它发出已启动通知；最后放行并等待清理结束。
gate_task = asyncio.create_task(wait_for_gate(started, gate, cleanup))
try:
    async with asyncio.timeout(3):
        await started.wait()
    print(gate_task.done())  # False：任务暂停，当前代码仍能运行。
finally:
    gate.set()
    print(await gate_task)  # 已放行
print(cleanup)  # ['已清理']

False
已放行
['已清理']


### 2.2 await 也可能立即完成

await 的操作若不需要挂起，可以在当前任务内直接完成；不能用“出现 await”推断已经切换任务。长时间同步计算、time.sleep 或同步 I/O 若直接运行在事件循环线程上，会阻塞该线程上的其他任务。

下面只做有限的小计算，并等待一个不包含挂起点的协程；观察另一任务必须等到当前任务主动让出后才能记录标记。同步 I/O 的迁移方法在后面的 to\_thread 示例中使用。

In [4]:
async def mark_progress(trace: list[str]) -> None:
    """记录后台任务获得执行机会。"""
    trace.append("后台执行")


async def return_immediately() -> int:
    """直接返回，不包含使当前任务挂起的操作。"""
    return 1


blocking_trace = []
marker = asyncio.create_task(mark_progress(blocking_trace))
try:
    calculation = sum(range(1000))  # 小型同步计算；不把它扩大成卡顿演示。
    immediate_value = await return_immediately()
    print(blocking_trace)  # []：这个 await 没有等待外部完成条件。
finally:
    await marker
print(blocking_trace)  # ['后台执行']
assert calculation == 499500 and immediate_value == 1

[]
['后台执行']


## 3 创建与收集任务

### 3.1 create\_task 后保留引用并等待

create\_task 把协程交给当前事件循环调度，并返回 Task。先创建多个任务，再逐个 await，可以让它们共同推进；逐个 await 已经创建的 Task，不会把创建过程变成逐个调用协程的顺序过程。

应保留任务引用并取得结果，不能把未完成任务随意留给下一个单元。下面用 TaskGroup 管理任务的退出等待；它的失败规则在下一节展开。

In [5]:
concurrent_trace = []
async with asyncio.TaskGroup() as group:
    label_tasks = [
        group.create_task(prepare_label(label, concurrent_trace))
        for label in ["a", "b"]
    ]
    labels = []
    for task in label_tasks:
        labels.append(await task)
print(labels)  # ['A', 'B']：按保存的任务顺序取结果。
assert all(task.done() for task in label_tasks)
assert set(concurrent_trace) == {"a开始", "a结束", "b开始", "b结束"}
# TaskGroup 退出前会等待其任务；不把具体的交错顺序作为业务前提。

['A', 'B']


### 3.2 gather 聚合多个结果

asyncio.gather 接收多个可等待对象；传入协程时会自动调度为任务，成功结果按传入顺序排列，而非完成顺序。默认遇到异常会向等待方传播，其他任务的处理规则需要另外考虑。

Task.result 不会像线程池 Future.result 那样阻塞等待；任务尚未完成时会抛出 InvalidStateError。因此先 await 或等待任务组退出，再读取 Task.result。

In [6]:
gather_trace = []
gathered_labels = await asyncio.gather(
    prepare_label("b", gather_trace),
    prepare_label("a", gather_trace),
)
print(gathered_labels)  # ['B', 'A']：顺序对应 gather 的两个参数。
assert gathered_labels == ["B", "A"]
print([task.result() for task in label_tasks])  # ['A', 'B']：前一单元已等待。

['B', 'A']
['A', 'B']


## 4 任务组合的失败行为

### 4.1 TaskGroup 等待整个任务组

TaskGroup 是异步上下文管理器（asynchronous context manager）：async with 退出时等待全部成员任务。任务的生命周期与代码块绑定，不需要把未结束的任务交给调用方猜测何时清理。

下面正常结束后才读取结果，任务数量固定为三项。async with 的进入和退出都允许等待，完整资源协议稍后介绍。

In [7]:
async def double_minutes(minutes: int) -> int:
    """让出执行机会后返回学习时长的两倍。"""
    await asyncio.sleep(0)
    return minutes * 2


async with asyncio.TaskGroup() as group:
    doubled_tasks = [
        group.create_task(double_minutes(minutes)) for minutes in [10, 20, 30]
    ]
print([task.result() for task in doubled_tasks])  # [20, 40, 60]
assert all(task.done() for task in doubled_tasks)

[20, 40, 60]


### 4.2 gather 默认不因一个失败而取消其他成员

默认 gather 会把第一个异常传播给等待方，其余成员继续运行。不能捕获错误后就忘掉这些成员；下面保留两个 Task，失败后显式放行并等待另一个。

return\_exceptions=True 会把异常对象混入结果列表，使用它时必须逐项检查，不能把异常当作普通成功结果。这里保持默认行为，只捕获明确的 ValueError。

In [8]:
async def fail_after_started(started: asyncio.Event) -> None:
    """确认另一任务已进入工作状态后，报告一条输入错误。"""
    await started.wait()
    raise ValueError("学习记录格式错误")


started = asyncio.Event()
gate = asyncio.Event()
gather_cleanup = []
survivor = asyncio.create_task(wait_for_gate(started, gate, gather_cleanup))
failure = asyncio.create_task(fail_after_started(started))
# 内层观察 gather 的失败传播；外层保证仍在等待的任务得到放行。
try:
    try:
        async with asyncio.timeout(3):
            await asyncio.gather(survivor, failure)
    except ValueError as error:
        print(type(error).__name__)  # ValueError
        print(survivor.done())  # False：默认 gather 没有取消其他成员。
    else:
        raise AssertionError("gather 没有传播预期异常")
finally:
    gate.set()
    await survivor
assert failure.done()
assert gather_cleanup == ["已清理"]

ValueError
False


### 4.3 TaskGroup 失败后取消其余成员并等待清理

组中第一个非 CancelledError 异常会触发其余任务的取消。TaskGroup 等待它们结束，再把非取消异常组合为异常组向外抛出；使用 except* 按异常类型处理。

KeyboardInterrupt 和 SystemExit 是特殊情况，清理后会重新抛出原异常。下面只演示 ValueError，且先用事件确保被取消任务已经进入 try/finally。

In [9]:
started = asyncio.Event()
gate = asyncio.Event()
group_cleanup = []
try:
    async with asyncio.timeout(3):
        async with asyncio.TaskGroup() as group:
            cancelled_member = group.create_task(
                wait_for_gate(started, gate, group_cleanup),
            )
            failing_member = group.create_task(fail_after_started(started))
except* ValueError as errors:
    print([type(error).__name__ for error in errors.exceptions])
    # ['ValueError']：同组取消不作为普通失败再次加入该异常组。
else:
    raise AssertionError("TaskGroup 没有产生预期异常组")
print(cancelled_member.cancelled(), group_cleanup)  # True ['已清理']
assert cancelled_member.cancelled()
assert group_cleanup == ["已清理"]
assert failing_member.done()

['ValueError']
True ['已清理']


## 5 异步迭代器与生成器

### 5.1 async for 每次等待下一项

异步可迭代对象的 \_\_aiter\_\_ 返回异步迭代器；\_\_anext\_\_ 返回一个可等待对象，完成时给出下一项，耗尽时抛出 StopAsyncIteration。async for 按次序等待每项，本身不会把循环体自动变成并发任务。

下面的类保留一个普通迭代器，但在每次取值前显式让出执行机会，用内存输入展示异步迭代协议。

In [10]:
class MinuteStream:
    """按异步迭代协议逐个提供内存中的学习时长。"""

    def __init__(self, minutes: list[int]) -> None:
        """创建只向前读取一次的内部迭代器。"""
        self._minutes = iter(minutes)

    def __aiter__(self) -> "MinuteStream":
        """返回当前异步迭代器。"""
        return self

    async def __anext__(self) -> int:
        """让出执行机会并取下一项，耗尽时结束异步迭代。"""
        await asyncio.sleep(0)
        try:
            return next(self._minutes)
        # 把同步迭代结束信号转换成异步迭代协议要求的结束信号。
        except StopIteration:
            raise StopAsyncIteration from None


minute_stream = MinuteStream([10, 20])
stream_minutes = []
async for minutes in minute_stream:
    stream_minutes.append(minutes)
print(stream_minutes)  # [10, 20]
try:
    await anext(minute_stream)
except StopAsyncIteration as error:
    print(type(error).__name__)  # StopAsyncIteration：已经耗尽。
else:
    raise AssertionError("耗尽的异步迭代器没有按预期结束")

[10, 20]
StopAsyncIteration


### 5.2 async def 中的 yield 创建异步生成器

async def 内包含 yield 时，定义的是异步生成器函数，调用得到异步生成器对象。它既能在 await 处等待，也能在 yield 处逐项交出值，不需要手写整个迭代器类。

下面在完整遍历结束时记录 finally；这里只模拟分批到达的内存记录，不代表实际网络或磁盘传输。

In [11]:
from collections.abc import AsyncIterator


async def generate_minutes(
    minutes: list[int],
    cleanup: list[str],
) -> AsyncIterator[int]:
    """逐项提供学习时长，并在生成器结束时记录清理。"""
    try:
        for value in minutes:
            await asyncio.sleep(0)
            yield value
    finally:
        cleanup.append("生成器关闭")


generator_cleanup = []
generated_minutes = []
async for minutes in generate_minutes([15, 25], generator_cleanup):
    generated_minutes.append(minutes)
print(generated_minutes, generator_cleanup)
# [15, 25] ['生成器关闭']：完整遍历到结束，finally 已执行。

[15, 25] ['生成器关闭']


### 5.3 提前 break 时显式关闭生成器

提前离开 async for 不应依赖将来的垃圾回收才执行生成器清理。contextlib.aclosing 在退出 async with 时等待对象的 aclose，使清理发生在本次使用范围内。

下面只消费第一项就 break，离开管理范围后检查清理标记。

In [12]:
import contextlib

early_cleanup = []
async with contextlib.aclosing(
    generate_minutes([10, 20, 30], early_cleanup),
) as minute_generator:
    async for minutes in minute_generator:
        print(minutes)  # 10：只消费一项。
        break
print(early_cleanup)  # ['生成器关闭']：无需等到内核关闭。
assert early_cleanup == ["生成器关闭"]

10
['生成器关闭']


## 6 async with 管理资源

async with 通过 \_\_aenter\_\_ 和 \_\_aexit\_\_ 等待进入与退出操作。contextlib.asynccontextmanager 可把恰好 yield 一次的异步生成器函数转换为管理器，yield 交出资源，finally 负责退出。

下面管理真实的内存文本流；关闭动作本身是同步的，让出操作仅用于展示退出阶段允许 await。业务错误在外层处理，资源先关闭，避免错误路径遗留打开的流。

In [13]:
import io


@contextlib.asynccontextmanager
async def text_session(
    text: str,
    events: list[str],
) -> AsyncIterator[io.StringIO]:
    """提供内存文本流，并在退出时关闭和记录资源状态。"""
    stream = io.StringIO(text)
    events.append("进入")
    # yield 把资源交给 async with；上下文退出后继续执行清理。
    try:
        yield stream
    finally:
        stream.close()
        await asyncio.sleep(0)
        events.append("退出")


session_events = []
try:
    async with text_session("25", session_events) as text_stream:
        print(text_stream.read())  # 25
        raise ValueError("读取后的业务错误")
except ValueError as error:
    print(type(error).__name__)  # ValueError：管理器没有吞掉业务异常。
else:
    raise AssertionError("业务错误没有传播到上下文外")
print(text_stream.closed, session_events)  # True ['进入', '退出']
assert text_stream.closed

25
ValueError
True ['进入', '退出']


## 7 超时与取消传播

### 7.1 在 timeout 管理范围外捕获 TimeoutError

asyncio.timeout 按秒限制等待，超时会取消当前任务，并在管理器退出时将相应 CancelledError 转成 TimeoutError。因此捕获 TimeoutError 的位置应在 async with 外。

这里使用 0 秒超时：到事件循环的下一次调度机会时触发取消。任务等待一个无人放行的事件，必然走超时路径；没有长时间休眠或网络依赖。超时处理依赖协作式取消，不能强制打断同步阻塞函数。

In [14]:
timeout_cleanup = []
never_released = asyncio.Event()
try:
    async with asyncio.timeout(0) as deadline:
        try:
            await never_released.wait()
        finally:
            timeout_cleanup.append("等待已退出")
except TimeoutError as error:
    print(type(error).__name__)  # TimeoutError：在管理器外捕获。
else:
    raise AssertionError("未放行的等待没有按预期超时")
print(deadline.expired(), timeout_cleanup)  # True ['等待已退出']
assert deadline.expired()
assert timeout_cleanup == ["等待已退出"]

TimeoutError
True ['等待已退出']


### 7.2 取消请求要传播，清理放在 finally

Task.cancel 请求在下一个合适机会向协程注入 asyncio.CancelledError；它不是立即终止。调用方仍需 await 被取消任务，确认退出和清理完成。

CancelledError 直接继承 BaseException。协程若显式捕获它，通常应在处理后重新抛出；吞掉取消会干扰 TaskGroup 和 timeout 的内部机制。下面的处理函数记录取消，再原样传播。

In [15]:
async def cancellable_read(
    started: asyncio.Event,
    events: list[str],
) -> None:
    """等待数据，在收到取消时传播异常并完成清理。"""
    started.set()
    try:
        await asyncio.Event().wait()
    # 记录取消后继续抛出；finally 随后释放资源，不把取消改成成功返回。
    except asyncio.CancelledError:
        events.append("收到取消")
        raise
    finally:
        events.append("资源释放")


started = asyncio.Event()
cancellation_events = []
read_task = asyncio.create_task(cancellable_read(started, cancellation_events))
try:
    async with asyncio.timeout(3):
        await started.wait()
finally:
    read_task.cancel()
    try:
        await read_task
    except asyncio.CancelledError as error:
        print(type(error).__name__)  # CancelledError：调用方确认了取消结束。
    else:
        raise AssertionError("任务没有传播取消")
print(read_task.cancelled(), cancellation_events)
# True ['收到取消', '资源释放']：清理完成后才离开这个单元。

CancelledError
True ['收到取消', '资源释放']


## 8 用 Semaphore 限制进入工作区的任务数

Semaphore 是信号量，维护可获得的名额数；async with 在进入时等待名额，退出时归还，包括异常退出。它限制同时进入受保护区域的任务数量，不限制已创建 Task 的总数；海量输入还需要分批生产或队列。

下面只创建四个任务，名额为两个。用事件把最先进入的两个任务暂时留在工作区，检查峰值恰好为二，再放行全部任务。计数修改之间没有 await，因此本例同一事件循环中的这些修改不会互相插入。

In [16]:
from dataclasses import dataclass


@dataclass
class Capacity:
    """记录当前使用中的名额和历史峰值。"""

    active: int = 0
    peak: int = 0


async def limited_job(
    semaphore: asyncio.Semaphore,
    gate: asyncio.Event,
    at_capacity: asyncio.Event,
    capacity: Capacity,
) -> None:
    """占用一个名额，收到放行后归还并更新计数。"""
    # 只有取得名额后才增加活动计数；等待名额的任务不计入 active。
    async with semaphore:
        capacity.active += 1
        capacity.peak = max(capacity.peak, capacity.active)
        # 达到两个并发名额时通知观察方，使峰值实验不依赖任务调度快慢。
        if capacity.active == 2:
            at_capacity.set()
        try:
            await gate.wait()
        finally:
            capacity.active -= 1


semaphore = asyncio.Semaphore(2)
gate = asyncio.Event()
at_capacity = asyncio.Event()
capacity = Capacity()
# 启动四个任务共享两个名额，退出 TaskGroup 时它们均已结束。
async with asyncio.TaskGroup() as group:
    limited_tasks = [
        group.create_task(limited_job(semaphore, gate, at_capacity, capacity))
        for _ in range(4)
    ]
    try:
        async with asyncio.timeout(3):
            await at_capacity.wait()
        print(capacity.active)  # 2：其他两项尚未进入工作区。
    finally:
        gate.set()
print(capacity.active, capacity.peak)  # 0 2：全部退出，峰值受限。
assert capacity == Capacity(active=0, peak=2)
assert all(task.done() for task in limited_tasks)

2
0 2


## 9 用 to\_thread 衔接同步函数

### 9.1 在线程中执行同步 I/O 接口

asyncio.to\_thread 返回协程，等待它即可在线程中调用普通函数并取得结果。它适合把原有同步 I/O 接口移出事件循环线程；CPython 3.12 中的纯 Python CPU 密集计算通常不能靠它绕过 GIL 加速。

下面使用很小的内存文本读取，观察接口衔接，不声称产生了 I/O 加速。with 在工作函数内管理资源，函数异常也会通过 await 返回给调用方。

In [17]:
def read_text_number(text: str) -> int:
    """用同步文本流读取一个整数，退出时关闭文本流。"""
    with io.StringIO(text) as stream:
        return int(stream.read())


print(await asyncio.to_thread(read_text_number, "25"))  # 25
try:
    await asyncio.to_thread(read_text_number, "无效")
except ValueError as error:
    print(type(error).__name__)  # ValueError：同步函数异常传播到等待方。
else:
    raise AssertionError("同步读取的错误没有传播")

25
ValueError


### 9.2 取消等待方不能强制停止底层线程

已经开始执行的线程函数不能被 asyncio 的取消请求强制中断。本例保留承载 to\_thread 的 Task，用 shield 阻止等待方的取消传给它，然后显式通知线程结束并 await 这个 Task，确认底层调用完成。

shield 只控制 Task 的取消传播，不为线程提供中断能力。线程从 threading.Event 收到放行；通知事件循环里的 asyncio.Event 时，通过 call\_soon\_threadsafe 安排回调，不从线程直接操作 asyncio 同步对象。线程的等待本身也设为有限 3 秒。

In [18]:
import threading


def read_after_release(
    release: threading.Event,
    loop: asyncio.AbstractEventLoop,
    started: asyncio.Event,
) -> tuple[str, bool]:
    """等到通知后读取文本，并在返回前关闭流。"""
    with io.StringIO("25") as stream:
        # 当前位于工作线程，通过事件循环安排通知，避免跨线程直接操作事件。
        loop.call_soon_threadsafe(started.set)
        if not release.wait(timeout=3):
            raise TimeoutError("读取线程未在 3 秒内收到通知")
        text = stream.read()
    return text, stream.closed


async def wait_for_thread(task: asyncio.Task) -> tuple[str, bool]:
    """等待保留的线程任务，避免调用方取消传到该任务。"""
    return await asyncio.shield(task)


release = threading.Event()
started = asyncio.Event()
thread_task = asyncio.create_task(asyncio.to_thread(
    read_after_release, release, asyncio.get_running_loop(), started,
))
# 保留底层任务，另建可取消的等待方，区分“取消等待”和“线程结束”。
thread_waiter = asyncio.create_task(wait_for_thread(thread_task))
try:
    async with asyncio.timeout(3):
        await started.wait()
    thread_waiter.cancel()
    try:
        await thread_waiter
    except asyncio.CancelledError as error:
        print(type(error).__name__)  # CancelledError：等待方已经取消。
    else:
        raise AssertionError("等待方没有按预期取消")
    print(thread_task.done())  # False：读取线程仍在等放行。
finally:
    release.set()
    thread_report = await thread_task
    if not thread_waiter.done():
        await thread_waiter
print(thread_report)  # ('25', True)：函数已返回，文本流已关闭。
assert thread_report == ("25", True)

CancelledError
False
('25', True)


## 10 用 contextvars 隔离任务上下文

### 10.1 用 Token 恢复原值

ContextVar 保存当前上下文里的值，适合请求标识等随调用链传播的信息。get 读取，set 返回一个 Token，reset(token) 恢复该次 set 前的状态；这里 token 表示恢复凭证。

ContextVar 应在模块顶层创建，不放在闭包里。Python 3.12 中用 try/finally 显式 reset，不把 Token 当作 with 管理器；上下文存放的是对象引用，不能把上下文复制理解成对可变值的深拷贝。

In [19]:
import contextvars

request_id = contextvars.ContextVar("request_id", default="未设置")
request_token = request_id.set("request-A")
try:
    print(request_id.get())  # request-A
finally:
    request_id.reset(request_token)
print(request_id.get())  # 未设置：恢复 set 之前的状态。

request-A
未设置


### 10.2 Task 默认复制创建时的上下文

create\_task 未指定 context 时，复制创建任务时的当前上下文；随后父任务重新 set，不会改写已经创建的子任务上下文。每个子任务自己的 set 和 reset 也只作用于自己的上下文。

to\_thread 会把调用它的协程所处的当前上下文传给工作线程，适合让同步日志读取同一请求标识。下面使用不可变字符串观察任务间隔离，事件保证读取发生在父任务更改之后。

In [20]:
async def inspect_request(
    label: str,
    gate: asyncio.Event,
) -> tuple[str, str, str]:
    """记录继承值、任务局部值和恢复值。"""
    inherited = request_id.get()
    # token 保存本任务修改前的值，退出时只恢复本任务的上下文。
    token = request_id.set(label)
    try:
        await gate.wait()
        local_value = request_id.get()
    finally:
        request_id.reset(token)
    return inherited, local_value, request_id.get()


gate = asyncio.Event()
parent_token = request_id.set("创建时")
try:
    async with asyncio.TaskGroup() as group:
        context_tasks = [
            group.create_task(inspect_request(label, gate))
            for label in ["A", "B"]
        ]
        # 子任务创建后再改父任务的值，用来观察创建时复制的上下文。
        later_token = request_id.set("父任务后来修改")
        try:
            gate.set()
            context_reports = []
            for task in context_tasks:
                context_reports.append(await task)
            print(request_id.get())  # 父任务后来修改：子任务 set 不影响它。
            print(await asyncio.to_thread(request_id.get))
            # 父任务后来修改：当前上下文传播到同步线程函数。
        finally:
            request_id.reset(later_token)
finally:
    request_id.reset(parent_token)
print(context_reports)
# [('创建时', 'A', '创建时'), ('创建时', 'B', '创建时')]
print(request_id.get())  # 未设置：父任务的两次 set 也已分别恢复。
assert context_reports == [("创建时", "A", "创建时"), ("创建时", "B", "创建时")]

父任务后来修改
父任务后来修改
[('创建时', 'A', '创建时'), ('创建时', 'B', '创建时')]
未设置


## 本章小结

（1）调用协程函数不自动运行，await 也不保证每次挂起；同步阻塞工作会占住事件循环线程。需要并发时创建任务并保留、等待它们。

（2）gather 按参数顺序聚合结果，默认单个失败不取消其他成员。TaskGroup 在成员失败后取消其余成员，等待清理，再传播异常组。

（3）async for 等待逐项生成，async with 等待进入和退出。提前离开异步生成器时显式 aclose，取消清理放在 finally，不能随意吞掉 CancelledError。

（4）Semaphore 限制工作区内任务数；to\_thread 承接同步 I/O，但取消等待不会强制停止线程。ContextVar 保存任务上下文，set 的恢复责任用 Token 和 finally 表达。

自查：出现超时时，能否指出谁被取消、谁仍在执行、谁负责释放资源，以及调用方在哪一步确认了清理完成？

## 练习

（1）先预测输出，再运行核对。说明为什么第二个列表对应协程函数的实际执行时机，而不是调用产生协程对象的时机。

In [21]:
exercise_trace = []
exercise_coroutine = read_minutes(exercise_trace)
print(exercise_trace)
exercise_minutes = await exercise_coroutine
print(exercise_trace, exercise_minutes)
# 先写下预测，再核对协程创建与 await 的区别；不留下未等待对象。

[]
['开始读取'] 25


（2）使用 TaskGroup 为 [5, 10, 15] 创建 double\_minutes 任务。检查结果按输入顺序为 [10, 20, 30]，离开组时所有任务 done；然后加入等待 started 的 ValueError 任务和等待 gate 的成员，检查异常组只包含预期失败，被取消成员恰好清理一次。

提示：可复用 fail\_after\_started 与 wait\_for\_gate；创建新的 Event 和清理列表，不依赖上文已经置位的事件。

In [22]:
exercise_minutes_list = [5, 10, 15]
# 先完成正常组，再创建独立事件运行失败组。
# 在 except* ValueError 外检查成员 cancelled 和清理标记，不吞掉其他异常。

（3）在 text\_session 中等待一个无人放行的 Event，用 asyncio.timeout(0) 触发超时。检查外层捕获 TimeoutError，文本流已关闭，进入和退出标记各一次。

再在这个操作外设置请求标识，并用 finally 恢复 Token；检查失败后 request\_id 回到原值。不要为使练习结束而吞掉内部 CancelledError。

In [23]:
exercise_session_events = []
exercise_request_before = request_id.get()
# 为本次操作 set 请求标识，构造超时和资源上下文，在最外层 reset。
# 检查流的 closed、事件列表和最终请求标识，不访问网络。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [协程函数、协程对象与执行入口](https://docs.python.org/3.12/library/asyncio-task.html#coroutines)、[可等待对象](https://docs.python.org/3.12/library/asyncio-task.html#awaitables)、[create\_task 的引用与上下文](https://docs.python.org/3.12/library/asyncio-task.html#asyncio.create_task)、[Task 的协作调度与 result](https://docs.python.org/3.12/library/asyncio-task.html#asyncio.Task)、[sleep(0) 的挂起](https://docs.python.org/3.12/library/asyncio-task.html#asyncio.sleep)、[gather 的结果顺序与失败行为](https://docs.python.org/3.12/library/asyncio-task.html#asyncio.gather)、[TaskGroup 的等待、取消与异常组](https://docs.python.org/3.12/library/asyncio-task.html#task-groups)、[取消传播和 finally](https://docs.python.org/3.12/library/asyncio-task.html#task-cancellation)、[timeout 与异常捕获位置](https://docs.python.org/3.12/library/asyncio-task.html#asyncio.timeout)、[to\_thread 的 I/O 与上下文边界](https://docs.python.org/3.12/library/asyncio-task.html#asyncio.to_thread)、[shield](https://docs.python.org/3.12/library/asyncio-task.html#asyncio.shield)；[事件循环中的阻塞与线程通知](https://docs.python.org/3.12/library/asyncio-dev.html#concurrency-and-multithreading)、[阻塞代码](https://docs.python.org/3.12/library/asyncio-dev.html#running-blocking-code)、[Event](https://docs.python.org/3.12/library/asyncio-sync.html#asyncio.Event)、[Semaphore](https://docs.python.org/3.12/library/asyncio-sync.html#asyncio.Semaphore)、[asyncio.run 的同线程循环限制](https://docs.python.org/3.12/library/asyncio-runner.html#asyncio.run)；[异步迭代协议](https://docs.python.org/3.12/reference/datamodel.html#asynchronous-iterators)、[异步生成器](https://docs.python.org/3.12/reference/expressions.html#asynchronous-generator-functions)、[异步上下文协议](https://docs.python.org/3.12/reference/datamodel.html#asynchronous-context-managers)、[asynccontextmanager](https://docs.python.org/3.12/library/contextlib.html#contextlib.asynccontextmanager)、[aclosing 的提前退出清理](https://docs.python.org/3.12/library/contextlib.html#contextlib.aclosing)；[ContextVar、默认值与顶层创建](https://docs.python.org/3.12/library/contextvars.html#contextvars.ContextVar)、[set](https://docs.python.org/3.12/library/contextvars.html#contextvars.ContextVar.set)、[reset](https://docs.python.org/3.12/library/contextvars.html#contextvars.ContextVar.reset)、[asyncio 上下文支持](https://docs.python.org/3.12/library/contextvars.html#asyncio-support)、[线程 Future 的取消边界](https://docs.python.org/3.12/library/concurrent.futures.html#concurrent.futures.Future.cancel)。 |
| IPython 官方文档（Autoawait） | [Notebook 顶层异步语法](https://ipython.readthedocs.io/en/stable/interactive/autoawait.html#using-autoawait-in-a-notebook-ipykernel)、[IPykernel 持续运行事件循环与终端差异](https://ipython.readthedocs.io/en/stable/interactive/autoawait.html#difference-between-terminal-ipython-and-ipykernel)。 |